# RecapAI - Meeting Summarizer
**Course:** CSC 603 - Generative AI | **Team:** Adrian Aquino, Charlie Huynh, Will Brust | **Spring 2026**

## Install Libraries

In [1]:
%pip install -q torch transformers accelerate huggingface_hub python-dotenv ipywidgets

print('Done!')

Note: you may need to restart the kernel to use updated packages.
Done!



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Libraries

In [ ]:
import os
import json
import torch
from transformers import pipeline
from huggingface_hub import login
from dotenv import load_dotenv, set_key

# if running from the notebooks folder, step up to project root so all paths work
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# only import colab files if running on Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Done!')

## Hugging Face Login

In [ ]:
if IN_COLAB:
    # on Colab: load from Colab Secrets (Secrets tab in left sidebar)
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    # locally: load from .env file
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

    if not HF_TOKEN:
        print('Enter your Hugging Face token when prompted, then press Enter.')
        HF_TOKEN = input('HF Token: ')
        set_key('.env', 'HF_TOKEN', HF_TOKEN)
        print('Token saved to .env')

login(token=HF_TOKEN)
print('Logged in!')

## Load the Model

In [ ]:
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

# use GPU if available, otherwise CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

llm = pipeline(
    task='text-generation',
    model=MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
)

print('Model loaded!')

## Summarize Function

In [ ]:
ROLE = "You are an AI assistant that summarizes meeting transcripts into structured JSON."

TASK = """Summarize the following meeting transcript.
Return ONLY a JSON object with exactly these 4 keys:
- "summary": a short 2-3 sentence overview of the meeting
- "decisions": a list of decisions that were made
- "assigned_tasks": a list of objects with "who", "what", and "due" fields
- "open_questions": a list of questions that were raised but not resolved

Do not include any explanation or text outside the JSON."""

# sends a single chunk to the model and returns parsed JSON
def summarize_chunk(transcript, max_tokens=1024):
    """Sends one transcript chunk to the model and returns parsed JSON.

    Args:
        transcript (str): A single chunk of transcript text.
        max_tokens (int): Max tokens to generate.

    Returns:
        dict: JSON output with summary, decisions, assigned_tasks, open_questions.
    """
    messages = [
        {'role': 'system', 'content': f'{ROLE}'},
        {'role': 'user',   'content': f'{TASK}\n\nTranscript:\n{transcript}'}
    ]

    print('Sending to model...')
    response = llm(
        messages,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=1.0
    )

    # pull text from response
    raw_output = response[0]['generated_text'][-1]['content']
    print('Response received. Parsing...')

    return try_parse_json(raw_output)

print('Function defined!')

## JSON Parser with Fallback

In [ ]:
def try_parse_json(raw_output):
    """Parses model output as JSON. Retries once on failure, then falls back to raw text.

    Args:
        raw_output (str): Raw text returned by the model.

    Returns:
        dict: Parsed JSON or a fallback dict with raw text under 'summary'.
    """
    # attempt 1: direct parse
    try:
        start  = raw_output.index('{')
        end    = raw_output.rindex('}') + 1
        parsed = json.loads(raw_output[start:end])
        print('Parsed on first attempt.')
        return parsed
    except (ValueError, json.JSONDecodeError):
        print('First attempt failed. Trying repair...')

    # attempt 2: ask model to fix its own output
    try:
        repair_messages = [
            {'role': 'system', 'content': 'You fix broken JSON. Return only valid JSON and nothing else.'},
            {'role': 'user',   'content': f'Fix this JSON:\n\n{raw_output}'}
        ]
        repair_response = llm(repair_messages, max_new_tokens=1024, do_sample=False, temperature=1.0)
        repaired        = repair_response[0]['generated_text'][-1]['content']
        start           = repaired.index('{')
        end             = repaired.rindex('}') + 1
        parsed          = json.loads(repaired[start:end])
        print('Parsed after repair.')
        return parsed
    except (ValueError, json.JSONDecodeError):
        print('Repair failed. Returning fallback.')

    # fallback: return raw text so something is shown instead of a crash
    return {
        'summary': raw_output,
        'decisions': [],
        'assigned_tasks': [],
        'open_questions': ['[Could not parse output - see summary for raw text]']
    }

print('Function defined!')

## Validate Output

In [ ]:
def validate_output(result):
    """Checks that all 4 required sections exist in the output.

    Args:
        result (dict): Parsed output from summarize_meeting().

    Returns:
        bool: True if all sections present, False if any are missing.
    """
    required    = ['summary', 'decisions', 'assigned_tasks', 'open_questions']
    all_present = True

    for section in required:
        if section not in result:
            print(f"WARNING: Missing -> '{section}'")
            all_present = False
        else:
            print(f"OK: '{section}'")

    print('\nAll good!' if all_present else '\nSome sections missing. Check the prompt.')
    return all_present

print('Function defined!')

## Chunking (for long transcripts)

In [ ]:
# max characters sent to the model per chunk
CHUNK_SIZE = 1500

def chunk_transcript(transcript, max_chars=CHUNK_SIZE):
    """Splits a transcript into smaller chunks by sentence boundary.

    Args:
        transcript (str): The full transcript text.
        max_chars (int): Max characters per chunk.

    Returns:
        list[str]: List of transcript chunks.
    """
    sentences = transcript.replace('\n', ' ').split('. ')
    chunks  = []
    current = ''

    for sentence in sentences:
        segment = sentence + '. '
        if len(current) + len(segment) > max_chars and current:
            chunks.append(current.strip())
            current = segment
        else:
            current += segment

    if current.strip():
        chunks.append(current.strip())

    return chunks


def removedupe(items):
    """Removes duplicates from a list while keeping the original order.

    Args:
        items (list): List of strings or dicts to deduplicate.

    Returns:
        list: Deduplicated list in original order.
    """
    seen   = set()
    result = []

    for item in items:
        # convert dicts to a string key so they can be checked in a set
        key = json.dumps(item, sort_keys=True) if isinstance(item, dict) else str(item)
        if key not in seen:
            seen.add(key)
            result.append(item)

    return result


def merge_results(results):
    """Combines per-chunk JSON summaries into one result, removing duplicates.

    Args:
        results (list[dict]): List of JSON outputs from summarize_chunk().

    Returns:
        dict: Single merged JSON output.
    """
    merged = {
        'summary':        '',
        'decisions':      [],
        'assigned_tasks': [],
        'open_questions': []
    }

    summaries = []
    for r in results:
        if r.get('summary'):
            summaries.append(r['summary'])
        merged['decisions']      += r.get('decisions', [])
        merged['assigned_tasks'] += r.get('assigned_tasks', [])
        merged['open_questions'] += r.get('open_questions', [])

    # join summaries and remove any duplicate items across chunks
    merged['summary']        = ' '.join(summaries)
    merged['decisions']      = removedupe(merged['decisions'])
    merged['assigned_tasks'] = removedupe(merged['assigned_tasks'])
    merged['open_questions'] = removedupe(merged['open_questions'])

    return merged


# main entry point — handles both short and long transcripts
def summarize_meeting(transcript, max_chars=CHUNK_SIZE):
    """Chunks a long transcript and merges results. Skips chunking if short enough.

    Args:
        transcript (str): The full transcript text.
        max_chars (int): Max characters per chunk.

    Returns:
        dict: Merged JSON output.
    """
    if len(transcript) <= max_chars:
        print(f'Short enough, no chunking needed ({len(transcript)} chars).')
        return summarize_chunk(transcript)

    chunks = chunk_transcript(transcript, max_chars)
    print(f'Split into {len(chunks)} chunks.')

    results = []
    for i, chunk in enumerate(chunks):
        print(f'\n--- Chunk {i + 1} of {len(chunks)} ({len(chunk)} chars) ---')
        results.append(summarize_chunk(chunk))

    print('\nMerging results...')
    return merge_results(results)


print('Chunking functions defined!')

## Input Transcript
Choose how to provide your transcript: paste text, upload a file, or generate a mock transcript using AI.

In [ ]:
print('How would you like to input the transcript?')
print('1. Paste plain text')
print('2. Upload a .txt file')
print('3. Generate mock transcripts using AI')

choice = input('Enter choice (1/2/3): ').strip()

if choice == '1':
    print('Enter your transcript when prompted, then press Enter.')
    TRANSCRIPT = input('Transcript: ')
    print(f'Transcript set! Length: {len(TRANSCRIPT)} characters')

elif choice == '2':
    if IN_COLAB:
        uploaded   = files.upload()
        filename   = list(uploaded.keys())[0]
        TRANSCRIPT = uploaded[filename].decode('utf-8')
    else:
        path = input('Enter path to your .txt file: ')
        with open(path, 'r') as f:
            TRANSCRIPT = f.read()
        filename = path
    print(f'Loaded: {filename}')
    print(f'Length: {len(TRANSCRIPT)} characters')

elif choice == '3':
    count = int(input('How many transcripts to generate? (1-20): ').strip())
    count = max(1, min(20, count))

    MOCK_PROMPT = """Generate a realistic work meeting transcript as raw spoken dialogue only.

Rules:
- No speaker labels (e.g. 'Mike:' or 'Dave said:')
- No metadata (date, title, attendees, meeting name)
- No headers or section dividers
- Write as one continuous block of text with no line breaks between statements
- Names can appear naturally in conversation (e.g. 'Hey Mike, did you finish that?')
- Natural conversational tone, include interruptions, filler words, and back-and-forth
- Output plain text only"""

    messages = [
        {'role': 'system', 'content': 'You generate realistic meeting transcripts.'},
        {'role': 'user',   'content': MOCK_PROMPT}
    ]

    # clear old generated transcripts before saving new ones
    os.makedirs('GeneratedMockTranscripts', exist_ok=True)
    for old_file in os.listdir('GeneratedMockTranscripts'):
        if old_file.startswith('GeneratedMockTranscript-') and old_file.endswith('.txt'):
            os.remove(f'GeneratedMockTranscripts/{old_file}')
    print('Cleared old transcripts.')

    for i in range(1, count + 1):
        print(f'Generating transcript {i} of {count}...')
        response       = llm(messages, max_new_tokens=800, do_sample=True, temperature=0.9)
        transcript_txt = response[0]['generated_text'][-1]['content']
        filename       = f'GeneratedMockTranscripts/GeneratedMockTranscript-{i:02d}.txt'
        with open(filename, 'w') as f:
            f.write(transcript_txt)
        print(f'Saved: {filename}')

    print(f'\nAll {count} transcripts saved!')

    # clear old mock summaries and summarize all new transcripts
    os.makedirs('Summaries', exist_ok=True)
    for old_file in os.listdir('Summaries'):
        if old_file.startswith('GeneratedMockTranscript-') and old_file.endswith('_summary.json'):
            os.remove(f'Summaries/{old_file}')

    print(f'\nSummarizing all {count} transcripts...')
    for i in range(1, count + 1):
        transcript_path = f'GeneratedMockTranscripts/GeneratedMockTranscript-{i:02d}.txt'
        with open(transcript_path, 'r') as f:
            transcript = f.read()
        print(f'\n--- Transcript {i} of {count} ---')
        result = summarize_meeting(transcript)
        out_name = f'Summaries/GeneratedMockTranscript-{i:02d}_summary.json'
        with open(out_name, 'w') as f:
            json.dump(result, f, indent=2)
        print(f'Saved: {out_name}')

    print(f'\nDone! All {count} transcripts summarized and saved to Summaries/')
    TRANSCRIPT = ''

else:
    print('Invalid choice. Run this cell again and enter 1, 2, or 3.')
    TRANSCRIPT = ''

## Run

In [ ]:
if not TRANSCRIPT.strip():
    print('Choice 3: transcripts already summarized and saved to Summaries/.')
else:
    result = summarize_meeting(TRANSCRIPT)

    os.makedirs('Output', exist_ok=True)
    with open('Output/output.json', 'w') as f:
        json.dump(result, f, indent=2)

    os.makedirs('Summaries', exist_ok=True)
    with open('Summaries/input_summary.json', 'w') as f:
        json.dump(result, f, indent=2)

    print('Done! Saved to Output/output.json and Summaries/input_summary.json')